# Unit 5 – Practice & Consolidation Worksheet

These exercises are for your own practice and are **not graded**. Work through them after completing the Unit 5 reading material. Attempt each exercise before revealing the answer.

## Exercise types

| Type | What to do |
|---|---|
| Predict the output | Write what you think the code will print *before* running the cell |
| Fix the bug | The code contains an error — find and correct it |
| Fill in the blank | Complete the missing line(s) to produce the expected output |
| Adapt and extend | A working example is given — modify it to solve a related problem |

**Topics:** train/test split and data leakage · feature scaling · imputation · encoding categorical features · basic pipelines · ColumnTransformer · end-to-end pipelines and inspection

---

## Section 1 – Train/Test Split and Data Leakage

### Exercise 1.1 Predict the output

Without running the cell, write down what will be printed.

```python
from sklearn.model_selection import train_test_split
import numpy as np

X = np.arange(20).reshape(10, 2)
y = np.array([0, 1, 0, 1, 0, 1, 0, 1, 0, 1])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

print(X_train.shape)
print(X_test.shape)
print(len(y_train))
```

<details>
<summary><strong>Select for answer</strong></summary>

```python
# (7, 2)
# (3, 2)
# 7
```

`test_size=0.3` puts 30% of the 10 samples into the test set: 10 × 0.3 = 3 test samples and 7 training samples. Both `X` splits retain the 2-column structure. `y_train` has 7 labels matching the 7 training rows.

</details>

### Exercise 1.2 Fill in the blank

Complete the split so that:
- 20% of data goes to the test set
- The class distribution is preserved in both sets (`stratify`)
- Results are reproducible

```python
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np

np.random.seed(42)
n = 200
X = pd.DataFrame({
    'age':    np.random.randint(22, 60, n),
    'salary': np.random.randint(25000, 90000, n),
    'years':  np.random.randint(0, 20, n)
})
y = pd.Series(np.random.choice(['Yes', 'No'], size=n, p=[0.3, 0.7]), name='promoted')

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=________,
    stratify=________,
    random_state=________
)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
```

<details>
<summary><strong>Select for answer</strong></summary>

```python
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)
```

`test_size=0.2` reserves 20% for testing. `stratify=y` ensures the proportion of each class label is the same in both splits — essential when one class is much rarer than the other. `random_state` can be any integer; the same value always produces the same split.

</details>

### Exercise 1.3 Fix the bug

The code below leaks data from the test set into the scaler. Identify the problem and fix it.

```python
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import numpy as np

X = np.random.randn(100, 4)
y = np.random.randint(0, 2, 100)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)          # ← leakage here

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)
```

<details>
<summary><strong>Select for answer</strong></summary>

```python
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # fit on training only
X_test_scaled  = scaler.transform(X_test)         # transform test with training statistics
```

Fitting the scaler on the entire dataset before splitting means the scaler's mean and standard deviation are computed using test set values — information the model should never see during training. The golden rule is: **split first, then fit**. The scaler should only call `.fit_transform()` on training data and `.transform()` (no fitting) on test data.

</details>

### Exercise 1.4 Predict the output

What will this code print, and what does the output tell you about the split?

```python
from sklearn.model_selection import train_test_split
import numpy as np

y = np.array([0, 0, 0, 0, 0, 0, 0, 1, 1, 1])   # 70% class 0, 30% class 1

y_train, y_test = train_test_split(y, test_size=0.4, stratify=y, random_state=0)

print("Train class 1 %:", y_train.mean().round(2))
print("Test class 1 %: ", y_test.mean().round(2))
```

<details>
<summary><strong>Select for answer</strong></summary>

```python
# Train class 1 %: 0.33
# Test class 1 %:  0.25
# (exact values may vary slightly due to rounding with small samples)
# Both should be close to 0.3 — the original proportion
```

`stratify=y` preserves the class ratio from the original array in both splits. With only 10 samples the proportions are approximate (you cannot split 3 ones exactly into 60/40), but they are much closer to 30% than an unstratified split would guarantee. This matters when one class is rare — without stratification the test set might contain no minority-class examples at all.

</details>

### Exercise 1.5 Adapt and extend

The example below performs a simple 80/20 split:

```python
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
```

Adapt it to create **three** sets: training (60%), validation (20%), and test (20%). Use two consecutive `train_test_split` calls.

```python
from sklearn.model_selection import train_test_split
import numpy as np

np.random.seed(0)
X = np.random.randn(100, 3)
y = np.random.randint(0, 2, 100)

# Your code here — print the size of each set
```

<details>
<summary><strong>Select for answer</strong></summary>

```python
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.4, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42
)

print(f'Train: {X_train.shape}')    # (60, 3)
print(f'Val:   {X_val.shape}')      # (20, 3)
print(f'Test:  {X_test.shape}')     # (20, 3)
```

The first split takes 60% for training and puts the remaining 40% into a temporary set. The second split divides that 40% equally: 50% of 40% = 20% validation and 20% test. The test set must never be used during model development — it is reserved for the final evaluation only.

</details>

---
## Section 2 – Feature Scaling

### Exercise 2.1 Predict the output

What will `StandardScaler` produce for this data?

```python
from sklearn.preprocessing import StandardScaler
import numpy as np

X = np.array([[10.0],
              [20.0],
              [30.0]])

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(X_scaled)
print(f"Mean after scaling: {X_scaled.mean():.1f}")
print(f"Std after scaling:  {X_scaled.std():.1f}")
```

<details>
<summary><strong>Select for answer</strong></summary>

```python
# [[-1.22474487]
#  [ 0.        ]
#  [ 1.22474487]]
# Mean after scaling: 0.0
# Std after scaling:  1.0
```

`StandardScaler` subtracts the mean (20) and divides by the standard deviation. The result always has mean ≈ 0 and standard deviation = 1. The middle value (20, which equals the mean) becomes exactly 0. The spacing between scaled values depends on the original spread.

</details>

### Exercise 2.2 Fill in the blank

Complete the code to apply `MinMaxScaler` correctly — fit on training data only, then transform both sets.

```python
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
import numpy as np

X = np.array([[5.0, 100.0],
              [10.0, 200.0],
              [15.0, 300.0],
              [20.0, 400.0],
              [25.0, 500.0]])
y = np.array([0, 0, 1, 1, 1])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.4, random_state=0)

scaler = MinMaxScaler()
X_train_scaled = scaler.________(X_train)
X_test_scaled  = scaler.________(X_test)

print("Train min:", X_train_scaled.min(axis=0).round(2))
print("Train max:", X_train_scaled.max(axis=0).round(2))
```

<details>
<summary><strong>Select for answer</strong></summary>

```python
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)
```

`fit_transform` on training data: learns the min and max from training rows, then applies the scaling. `transform` on test data: applies the **same** min and max learned from training — never refit on the test set. Test values may fall outside [0, 1] if they are outside the training range; that is expected and correct.  This ensures no information from the test set leaks into the model.

</details>

### Exercise 2.3 Fix the bug

The code fits the scaler twice — once on training data and once on test data. Fix it.

```python
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import train_test_split
import numpy as np

X_train = np.array([[1.0, 200.0], [2.0, 300.0], [3.0, 400.0]])
X_test  = np.array([[4.0, 500.0], [5.0, 600.0]])

scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.fit_transform(X_test)    # ← bug
```

<details>
<summary><strong>Select for answer</strong></summary>

```python
X_test_scaled = scaler.transform(X_test)
```

Calling `fit_transform` on the test set refits the scaler using test statistics (median and IQR of the test data). This is data leakage and also means the test data is scaled differently from the training data, so the model receives inconsistent inputs. Use `.transform()` — which applies the already-learned statistics — for all data after the initial fit.

</details>

### Exercise 2.4 Predict the output

What will the two column means and standard deviations be after scaling? The two features have very different ranges.

```python
from sklearn.preprocessing import StandardScaler
import numpy as np

X = np.array([[1.0,   1000.0],
              [2.0,   2000.0],
              [3.0,   3000.0],
              [4.0,   4000.0]])

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(X_scaled.mean(axis=0).round(1))
print(X_scaled.std(axis=0).round(1))
```

<details>
<summary><strong>Select for answer</strong></summary>

```python
# [0. 0.]
# [1. 1.]
```

Regardless of the original scale, `StandardScaler` always produces columns with mean 0 and standard deviation 1. A feature ranging from 1 to 4 gets the same treatment as one ranging from 1000 to 4000. This is why scaling matters for distance-based algorithms — without it, the large-scale feature would dominate.

</details>

### Exercise 2.5 Adapt and extend

The example below scales a single numeric column:

```python
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_train[['age']])
```

Adapt it to scale three numeric columns (`'age'`, `'hours_per_week'`, `'annual_salary'`) from a DataFrame. After scaling the training set, use `scaler.mean_` and `scaler.scale_` to print what the scaler learned for each feature.

```python
from sklearn.preprocessing import StandardScaler
import pandas as pd
import numpy as np

np.random.seed(1)
X_train = pd.DataFrame({
    'age':            np.random.randint(22, 60, 80),
    'hours_per_week': np.random.randint(30, 60, 80),
    'annual_salary':  np.random.randint(25000, 90000, 80)
})

numeric_cols = ['age', 'hours_per_week', 'annual_salary']

# Your code here
```

<details>
<summary><strong>Select for answer</strong></summary>

```python
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train[numeric_cols])

for col, mean, scale in zip(numeric_cols, scaler.mean_, scaler.scale_):
    print(f'{col}: mean={mean:.1f}, std={scale:.1f}')
```

`scaler.mean_` and `scaler.scale_` store the statistics learned during `.fit()` — one value per feature in the order they were passed. These are the exact values that will be applied when `.transform()` is called on the test set. Inspecting them is a useful sanity check: the means should match your training data's column means.

</details>

---
## Section 3 – Imputation

### Exercise 3.1 Predict the output

What will `SimpleImputer` fill in for the missing values?

```python
from sklearn.impute import SimpleImputer
import numpy as np

X = np.array([[1.0, np.nan],
              [2.0, 3.0],
              [np.nan, 6.0],
              [4.0, 9.0]])

imputer = SimpleImputer(strategy='mean')
X_filled = imputer.fit_transform(X)

print(X_filled)
```

<details>
<summary><strong>Select for answer</strong></summary>

```python
# [[1.   6. ]
#  [2.   3. ]
#  [2.33 6. ]
#  [4.   9. ]]
# (column 0 mean = (1+2+4)/3 = 2.33, column 1 mean = (3+6+9)/3 = 6.0)
```

`SimpleImputer` computes the mean of each column **ignoring NaN values**, then substitutes that mean wherever a value is missing. Column 0 has three non-missing values: 1, 2, 4 → mean = 2.33. Column 1 has three non-missing values: 3, 6, 9 → mean = 6. The imputer fills each column independently.

</details>

### Exercise 3.2 Fill in the blank

Complete the code to impute numeric columns with the **median** and a categorical column with the **most frequent** value. Fit both imputers on training data only.

```python
from sklearn.impute import SimpleImputer
import pandas as pd
import numpy as np

X_train = pd.DataFrame({
    'salary':     [50000, np.nan, 60000, 70000, np.nan],
    'experience': [2, 5, np.nan, 8, 3],
    'department': ['Sales', None, 'Engineering', 'Sales', None]
})

# Numeric imputer
num_imputer = SimpleImputer(strategy=________)
num_imputer.________(X_train[['salary', 'experience']])

# Categorical imputer
cat_imputer = SimpleImputer(strategy=________)
cat_imputer.________(X_train[['department']])

print("Numeric fill values:", num_imputer.statistics_)
print("Categorical fill value:", cat_imputer.statistics_)
```

<details>
<summary><strong>Select for answer</strong></summary>

```python
num_imputer = SimpleImputer(strategy='median')
num_imputer.fit(X_train[['salary', 'experience']])

cat_imputer = SimpleImputer(strategy='most_frequent')
cat_imputer.fit(X_train[['department']])
```

`strategy='median'` is more robust than `'mean'` when the data has outliers — a single very high salary would pull the mean upward but barely affect the median. `strategy='most_frequent'` works for strings and categories. Both imputers only call `.fit()` here (not `.fit_transform()`) to learn the fill values without yet applying them.

</details>

### Exercise 3.3 Fix the bug

The code fits the imputer on the test set instead of the training set. Fix it.

```python
from sklearn.impute import SimpleImputer
import numpy as np

X_train = np.array([[1.0, np.nan], [2.0, 3.0], [3.0, 6.0]])
X_test  = np.array([[np.nan, 9.0], [4.0, np.nan]])

imputer = SimpleImputer(strategy='median')
X_train_filled = imputer.transform(X_train)    # ← bug
X_test_filled  = imputer.fit_transform(X_test)
```

<details>
<summary><strong>Select for answer</strong></summary>

```python
imputer = SimpleImputer(strategy='median')
X_train_filled = imputer.fit_transform(X_train)  # fit on training data
X_test_filled  = imputer.transform(X_test)         # apply same fill values
```

The imputer must `.fit()` on training data — that is where it learns the fill values (medians). Then `.transform()` applies those same values to both training and test data. Here the code never fits on training data at all: calling `.transform()` before `.fit()` raises a `NotFittedError`.

</details>

### Exercise 3.4 Predict the output

Predict what `imputer.statistics_` will contain and what the filled array will look like.

```python
from sklearn.impute import SimpleImputer
import numpy as np

X = np.array([[1.0, 10.0],
              [2.0, np.nan],
              [3.0, 30.0],
              [100.0, 40.0]])   # 100 is an outlier in column 0

imputer_mean   = SimpleImputer(strategy='mean')
imputer_median = SimpleImputer(strategy='median')

imputer_mean.fit(X)
imputer_median.fit(X)

print("Mean fill values:  ", imputer_mean.statistics_.round(1))
print("Median fill values:", imputer_median.statistics_.round(1))
```

<details>
<summary><strong>Select for answer</strong></summary>

```python
# Mean fill values:   [26.5 26.7]
# Median fill values: [ 2.5 30. ]
# Column 0 non-null values: 1, 2, 3, 100 → mean=26.5, median=2.5
# Column 1 non-null values: 10, 30, 40 → mean=26.7, median=30.0
```

The outlier (100) inflates the mean of column 0 to 26.5, which is far above most values. The median (2.5) is resistant to this outlier. For column 1, the missing value is ignored and the statistics are computed from the three present values. This demonstrates why `strategy='median'` is often preferred in the presence of skewed data or outliers.

</details>

### Exercise 3.5 Adapt and extend

The example below imputes a single column with the mean:

```python
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(strategy='mean')
X_train_filled = imputer.fit_transform(X_train[['age']])
```

Adapt it to handle a DataFrame with both numeric and categorical columns. Impute `'age'` and `'salary'` with the median, and `'job_type'` with the most frequent value. Then recombine the results into a single cleaned DataFrame.

```python
from sklearn.impute import SimpleImputer
import pandas as pd
import numpy as np

X_train = pd.DataFrame({
    'age':      [25, np.nan, 35, 42, np.nan],
    'salary':   [30000, 45000, np.nan, 60000, 55000],
    'job_type': ['Full-time', None, 'Part-time', 'Full-time', None]
})

# Your code here
```

<details>
<summary><strong>Select for answer</strong></summary>

```python
num_imputer = SimpleImputer(strategy='median')
cat_imputer = SimpleImputer(strategy='most_frequent')

num_filled = num_imputer.fit_transform(X_train[['age', 'salary']])
cat_filled = cat_imputer.fit_transform(X_train[['job_type']])

X_clean = pd.DataFrame(
    num_filled,
    columns=['age', 'salary']
)
X_clean['job_type'] = cat_filled

print(X_clean)
print('\nMissing values:', X_clean.isnull().sum().sum())
```

Numeric and categorical columns need separate imputers because the strategies are different. The outputs of `fit_transform` are NumPy arrays, so you reconstruct a DataFrame by passing the arrays and column names explicitly. In a full pipeline, `ColumnTransformer` handles this separation and recombination automatically — which is the topic of Section 6.

</details>

---
## Section 4 – Encoding Categorical Features

### Exercise 4.1 Predict the output

What will the encoded values be?

```python
from sklearn.preprocessing import OrdinalEncoder
import numpy as np

encoder = OrdinalEncoder(categories=[['Low', 'Medium', 'High', 'Very High']])

X_train = np.array([['High'], ['Low'], ['Medium'], ['High']])
X_encoded = encoder.fit_transform(X_train)

print(X_encoded.flatten())
```

<details>
<summary><strong>Select for answer</strong></summary>

```python
# [2. 0. 1. 2.]
```

`OrdinalEncoder` maps each category to its position in the provided list: `Low→0`, `Medium→1`, `High→2`, `Very High→3`. Providing the categories explicitly with `categories=[...]` ensures the order reflects the true ordering — without this, the encoder would sort alphabetically, which would make `High→0`, `Low→1`, `Medium→2` (wrong order).

</details>

### Exercise 4.2 Fill in the blank

Complete the code to apply `OneHotEncoder` to the `'city'` column. The encoder should silently ignore any unseen categories at prediction time rather than raising an error.

```python
from sklearn.preprocessing import OneHotEncoder
import numpy as np

X_train = np.array([['London'], ['Paris'], ['Berlin'], ['London']])
X_test  = np.array([['Paris'], ['Tokyo']])   # 'Tokyo' was never seen in training

ohe = OneHotEncoder(handle_unknown=________)
ohe.________(X_train)

X_train_enc = ohe.transform(X_train)
X_test_enc  = ohe.transform(X_test)

print("Categories learned:", ohe.categories_)
print("Test encoded:\n", X_test_enc.toarray())
```

<details>
<summary><strong>Answer</strong></summary>

```python
ohe = OneHotEncoder(handle_unknown='ignore')
ohe.fit(X_train)
```

`handle_unknown='ignore'` causes the encoder to produce an all-zeros row for any category not seen during training (here, 'Tokyo'). Without this, calling `.transform()` on unseen categories raises a `ValueError`. The test row for 'Paris' will be encoded normally; 'Tokyo' will produce all zeros — no column is activated for it.

</details>

### Exercise 4.3 Fix the bug

The code uses `OrdinalEncoder` for a nominal (unordered) feature. Fix it by using the correct encoder.

```python
from sklearn.preprocessing import OrdinalEncoder
import numpy as np

# 'colour' has no natural ordering — red is not 'more' than blue
X_train = np.array([['red'], ['blue'], ['green'], ['red']])
X_test  = np.array([['green'], ['blue']])

encoder = OrdinalEncoder()
encoder.fit(X_train)

X_train_enc = encoder.transform(X_train)
print(X_train_enc.flatten())   # [2. 0. 1. 2.] — implies red > green > blue, which is wrong
```

<details>
<summary><strong>Select for answer</strong></summary>

```python
from sklearn.preprocessing import OneHotEncoder
import numpy as np

X_train = np.array([['red'], ['blue'], ['green'], ['red']])
X_test  = np.array([['green'], ['blue']])

encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
encoder.fit(X_train)

X_train_enc = encoder.transform(X_train)
print(X_train_enc)
```

`OrdinalEncoder` assigns integers (0, 1, 2, ...) implying a numerical relationship between categories. A model trained on these values might learn that 'red' (2) is twice 'blue' (0), which is meaningless. `OneHotEncoder` creates a separate binary column for each category, making no ordinal assumptions. Use `OrdinalEncoder` only when the categories have a genuine order (e.g. Low < Medium < High).

</details>

### Exercise 4.4 Predict the output

How many columns will the encoded output have?

```python
from sklearn.preprocessing import OneHotEncoder
import numpy as np

X = np.array([['cat', 'S'],
              ['dog', 'M'],
              ['cat', 'L'],
              ['bird', 'S'],
              ['dog', 'M']])

ohe = OneHotEncoder(sparse_output=False)
X_enc = ohe.fit_transform(X)

print(X_enc.shape)
print(ohe.categories_)
```

<details>
<summary><strong>Select for answer</strong></summary>



```python
# (5, 6)
# [array(['bird', 'cat', 'dog'], dtype=object), array(['L', 'M', 'S'], dtype=object)]
```

Column 0 (`animal`) has 3 unique categories → 3 new columns.  
Column 1 (`size`) has 3 unique categories → 3 new columns.  
With the default `drop=None`, all categories are retained, giving a total of 3 + 3 = **6 columns** and an output shape of **(5, 6)**.

>If `drop='first'` were used, one column per feature would be dropped, giving a shape of `(5, 4)` instead.
</details>

### Exercise 4.5 Adapt and extend

The example below applies `OrdinalEncoder` to an education column:

```python
from sklearn.preprocessing import OrdinalEncoder
enc = OrdinalEncoder(categories=[['High School', 'Bachelor', 'Master', 'PhD']])
enc.fit(X_train[['education']])
```

Adapt it to encode a `'satisfaction'` column with levels `['Very Dissatisfied', 'Dissatisfied', 'Neutral', 'Satisfied', 'Very Satisfied']`. Then apply it to the training and test sets and verify that the mapping is correct by printing the unique encoded values alongside the original categories.

```python
from sklearn.preprocessing import OrdinalEncoder
import numpy as np

X_train = np.array([['Satisfied'], ['Neutral'], ['Very Satisfied'], ['Dissatisfied']])
X_test  = np.array([['Neutral'], ['Very Dissatisfied']])

# Your code here
```

<details>
<summary><strong>Select for answer</strong></summary>

```python
enc = OrdinalEncoder(
    categories=[['Very Dissatisfied', 'Dissatisfied', 'Neutral', 'Satisfied', 'Very Satisfied']]
)
enc.fit(X_train)

X_train_enc = enc.transform(X_train)
X_test_enc  = enc.transform(X_test)

print('Train encoded:', X_train_enc.flatten())   # [3. 2. 4. 1.]
print('Test encoded: ', X_test_enc.flatten())    # [2. 0.]
```

The categories list defines the order: index 0 is the lowest, index 4 is the highest. 'Very Dissatisfied'→0, 'Dissatisfied'→1, 'Neutral'→2, 'Satisfied'→3, 'Very Satisfied'→4. If 'Very Dissatisfied' appears in the test set but not in the training data, it is still encoded correctly because the mapping was defined explicitly — not learned from the training data values.

</details>

---
## Section 5 – Basic Pipelines

### Exercise 5.1 Predict the output

What will be printed, and what does each line tell you about the pipeline?

```python
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model',  LogisticRegression())
])

print(type(pipe))
print(pipe.steps)
print(len(pipe.steps))
print(pipe.named_steps.keys())
```

<details>
<summary><strong>Select for answer</strong></summary>

```python
# <class 'sklearn.pipeline.Pipeline'>
# [('scaler', StandardScaler()), ('model', LogisticRegression())]
# 2
# dict_keys(['scaler', 'model'])
```

`pipe.steps` returns the list of `(name, estimator)` tuples exactly as defined. `pipe.named_steps` is a dictionary keyed by the name strings — useful for accessing individual step objects after the pipeline has been fitted. `len(pipe.steps)` counts the number of steps.

</details>

### Exercise 5.2 Fill in the blank

Complete the code to build and train a two-step pipeline: scale features then fit a logistic regression model. Then use the pipeline to make predictions on the test set.

```python
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

pipe = Pipeline([
    ('scaler', ________),
    ('model',  ________)
])

pipe.________(X_train, y_train)

accuracy = pipe.________(X_test, y_test)
predictions = pipe.________(X_test)

print(f"Accuracy: {accuracy:.3f}")
print(f"First 5 predictions: {predictions[:5]}")
```

<details>
<summary><strong>Select for answer</strong></summary>

```python
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model',  LogisticRegression())
])

pipe.fit(X_train, y_train)

accuracy    = pipe.score(X_test, y_test)
predictions = pipe.predict(X_test)
```

Calling `pipe.fit(X_train, y_train)` runs the steps in order: `StandardScaler.fit_transform(X_train)` is called first, and the scaled output is passed to `LogisticRegression.fit(...)`. When `pipe.predict(X_test)` is called, the scaler applies `.transform()` (not `.fit_transform()`) before passing to the model — the test data is never used to update the scaler.

</details>

### Exercise 5.3 Fix the bug

The pipeline is built correctly but the code then scales the test data separately before predicting, which defeats the purpose of the pipeline. Fix it.

```python
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model',  LogisticRegression())
])
pipe.fit(X_train, y_train)

scaler = StandardScaler()
X_test_scaled = scaler.fit_transform(X_test)      # ← bug
predictions = pipe.predict(X_test_scaled)
```

<details>
<summary><strong>Select for answer</strong></summary>

```python
predictions = pipe.predict(X_test)
```

The pipeline already applies the scaler internally when `predict` is called — passing pre-scaled data means the data gets scaled twice, producing wrong inputs to the model. Additionally, creating a new `StandardScaler` and fitting it on the test set is itself a leakage problem. Pass raw test data directly to `pipe.predict()` and let the pipeline handle all preprocessing.

</details>

### Exercise 5.4 Predict the output

After fitting the pipeline, what will these inspection calls return?

```python
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model',  LogisticRegression(max_iter=200))
])
pipe.fit(X_train, y_train)

print(type(pipe.named_steps['scaler']))
print(pipe.named_steps['scaler'].mean_.round(2))
print(pipe.named_steps['model'].classes_)
```

<details>
<summary><strong>Select for answer</strong></summary>

```python
# <class 'sklearn.preprocessing.StandardScaler'>
# [mean of each feature from training data, e.g. ~5.83, ~3.07, ~3.77, ~1.21]
# [0 1 2]
```

After `.fit()`, each step in the pipeline is a fully fitted estimator. `pipe.named_steps['scaler']` returns the fitted `StandardScaler` object, whose `.mean_` attribute holds the per-feature means learned from `X_train`. `pipe.named_steps['model'].classes_` shows the class labels the classifier was trained on.

</details>

### Exercise 5.5 Adapt and extend

The example below builds a two-step pipeline:

```python
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model',  LogisticRegression())
])
```

Adapt it to add an intermediate PCA step (reducing to 2 components) between the scaler and the model. After fitting, use `pipe[:-1].transform(X_test)` to get the data after PCA but before classification, and print its shape.

```python
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Your code here
```

<details>
<summary><strong>Select for answer</strong></summary>

```python
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('pca',    PCA(n_components=2)),
    ('model',  LogisticRegression())
])

pipe.fit(X_train, y_train)

X_transformed = pipe[:-1].transform(X_test)
print(X_transformed.shape)   # (30, 2)
print(f'Accuracy: {pipe.score(X_test, y_test):.3f}')
```

`pipe[:-1]` selects all steps except the last (the classifier), creating a sub-pipeline. Calling `.transform()` on this sub-pipeline applies the scaler and PCA in sequence, returning the 2D representation of the test data. This slicing syntax is useful for visualising or debugging intermediate transformations without reconstructing the pipeline manually.

</details>

---
## Section 6 – ColumnTransformer

### Exercise 6.1 Predict the output

What will the transformed shape be, and why?

```python
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
import pandas as pd

df = pd.DataFrame({
    'age':    [25, 30, 35],
    'salary': [40000, 60000, 80000],
    'city':   ['London', 'Paris', 'Berlin']
})

preprocessor = ColumnTransformer([
    ('num', StandardScaler(),              ['age', 'salary']),
    ('cat', OneHotEncoder(sparse_output=False), ['city'])
])

X_out = preprocessor.fit_transform(df)
print(X_out.shape)
```

<details>
<summary><strong>Select for answer</strong></summary>

```python
# (3, 5)
# 2 numeric columns (scaled) + 3 one-hot columns (London/Paris/Berlin) = 5
```

`ColumnTransformer` applies each transformer to its specified columns and concatenates the results horizontally. `StandardScaler` on 2 numeric columns produces 2 columns. `OneHotEncoder` on `'city'` (3 unique values) produces 3 binary columns. The output has 2 + 3 = 5 columns and retains all 3 rows.

</details>

### Exercise 6.2 Fill in the blank

Complete the `ColumnTransformer` to apply:
- median imputation then standard scaling to numeric columns
- most-frequent imputation then one-hot encoding to categorical columns

```python
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

numeric_features     = ['age', 'salary']
categorical_features = ['department']

numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy=________)),
    ('scaler',  StandardScaler())
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy=________)),
    ('onehot',  OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', ________, ________),
    ('cat', ________, ________)
])

print(preprocessor)
```

<details>
<summary><strong>Select for answer</strong></summary>

```python
numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot',  OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])
```

Each transformer in the `ColumnTransformer` is a `(name, transformer, columns)` tuple. The transformer can be a single estimator or a nested `Pipeline` — nesting is the standard pattern when a column type needs more than one preprocessing step. The `ColumnTransformer` handles the splitting and recombining of columns automatically.

</details>

### Exercise 6.3 Fix the bug

The `ColumnTransformer` is constructed but the column names are swapped — numeric columns are passed to the categorical transformer and vice versa. Fix it.

```python
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

numeric_features     = ['age', 'income']
categorical_features = ['gender', 'region']

preprocessor = ColumnTransformer([
    ('num', StandardScaler(),              categorical_features),   # ← bug
    ('cat', OneHotEncoder(sparse_output=False), numeric_features)   # ← bug
])
```

<details>
<summary><strong>Select for answer</strong></summary>

```python
preprocessor = ColumnTransformer([
    ('num', StandardScaler(),                   numeric_features),
    ('cat', OneHotEncoder(sparse_output=False), categorical_features)
])
```

Applying `StandardScaler` to string columns would raise a `ValueError` because standard scaling requires numeric input. Applying `OneHotEncoder` to numeric columns would treat each distinct number as a separate category, producing a very wide output and losing the numerical meaning. Always match each transformer to the correct column list.

</details>

### Exercise 6.4 Predict the output

What is the difference between `remainder='drop'` and `remainder='passthrough'`?

```python
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
import pandas as pd

df = pd.DataFrame({
    'age':    [25, 30, 35],
    'salary': [40000, 60000, 80000],
    'id':     [101, 102, 103]          # we do not want to transform this
})

ct_drop = ColumnTransformer(
    [('scaler', StandardScaler(), ['age', 'salary'])],
    remainder='drop'
)

ct_pass = ColumnTransformer(
    [('scaler', StandardScaler(), ['age', 'salary'])],
    remainder='passthrough'
)

print(ct_drop.fit_transform(df).shape)
print(ct_pass.fit_transform(df).shape)
```

<details>
<summary><strong>Select for answer</strong></summary>

```python
# (3, 2)  — 'id' column dropped
# (3, 3)  — 'id' column appended as-is
```

`remainder='drop'` (the default) silently discards any columns not listed in a transformer. `remainder='passthrough'` appends untransformed columns to the right of the output. Use `'passthrough'` when you need to keep columns like IDs or already-encoded features. Use `'drop'` to discard irrelevant columns such as identifiers that should not be passed to the model.

</details>

### Exercise 6.5 Adapt and extend

The example below creates a basic `ColumnTransformer` for one numeric and one categorical column:

```python
preprocessor = ColumnTransformer([
    ('num', StandardScaler(),    ['age']),
    ('cat', OneHotEncoder(),     ['city'])
])
```

Adapt it to handle three feature types:
- `'age'` and `'salary'`: impute with median, then scale
- `'city'`: impute with most frequent, then one-hot encode  
- `'priority'` (ordinal: `'Low' < 'Medium' < 'High'`): ordinal encode

Then fit it on the training DataFrame below and print the output shape.

```python
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer
import pandas as pd
import numpy as np

X_train = pd.DataFrame({
    'age':      [25, np.nan, 35, 42],
    'salary':   [30000, 45000, np.nan, 60000],
    'city':     ['London', None, 'Paris', 'Berlin'],
    'priority': ['High', 'Low', 'Medium', 'High']
})

# Your code here
```

<details>
<summary><strong>Select for answer</strong></summary>

```python
num_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])

cat_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot',  OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

ord_transformer = OrdinalEncoder(categories=[['Low', 'Medium', 'High']])

preprocessor = ColumnTransformer([
    ('num', num_transformer,  ['age', 'salary']),
    ('cat', cat_transformer,  ['city']),
    ('ord', ord_transformer,  ['priority'])
])

X_out = preprocessor.fit_transform(X_train)
print(X_out.shape)  # (4, 6): 2 numeric + 3 one-hot (London/Paris/Berlin) + 1 ordinal
```

The ordinal transformer does not need imputation here because `'priority'` has no missing values — you can use the encoder directly without wrapping it in a `Pipeline`. The three transformers each handle different semantics: magnitude (scale), membership (one-hot), and order (ordinal). Combining them in one `ColumnTransformer` ensures a single, reproducible fit.

</details>

---
## Section 7 – End-to-End Pipelines and Inspection

### Exercise 7.1 Predict the output

What happens when `full_pipeline.fit(X_train, y_train)` is called? Describe the order of operations.

```python
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer

numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer,            ['age', 'salary']),
    ('cat', OneHotEncoder(sparse_output=False), ['department'])
])

full_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier',   LogisticRegression())
])

# Assume X_train, y_train exist
# full_pipeline.fit(X_train, y_train)

print(full_pipeline.steps)
print(len(full_pipeline.steps))
```

<details>
<summary><strong>Select for answer</strong></summary>

```python
# [('preprocessor', ColumnTransformer(...)), ('classifier', LogisticRegression())]
# 2

# Order of operations during .fit():
# 1. preprocessor.fit_transform(X_train):
#    a. SimpleImputer fills missing values in numeric columns (learned from X_train)
#    b. StandardScaler scales numeric columns (learned from X_train)
#    c. OneHotEncoder encodes department column (learned from X_train)
#    d. ColumnTransformer concatenates the outputs horizontally
# 2. LogisticRegression.fit(X_transformed, y_train)
```

The outer `Pipeline` has two steps: `'preprocessor'` and `'classifier'`. During `.fit()`, the preprocessor's `.fit_transform()` is called first, producing the fully transformed feature matrix. This is then passed to the classifier's `.fit()`. During `.predict()`, the preprocessor uses `.transform()` (not `.fit_transform()`) — no re-fitting occurs on new data.

</details>

### Exercise 7.2 Fill in the blank

Complete the code to build a full end-to-end pipeline for the Titanic dataset, then access the imputer's learned fill values via `named_steps`.

```python
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
import seaborn as sns

df = sns.load_dataset('titanic')[['age', 'fare', 'sex', 'embarked', 'survived']].dropna(subset=['survived'])

X = df.drop('survived', axis=1)
y = df['survived']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

numeric_features     = ['age', 'fare']
categorical_features = ['sex', 'embarked']

numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot',  OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', ________, numeric_features),
    ('cat', ________, categorical_features)
])

full_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier',   RandomForestClassifier(n_estimators=100, random_state=42))
])

full_pipeline.________(X_train, y_train)

print(f"Test accuracy: {full_pipeline.________(X_test, y_test):.3f}")

# Access the imputer inside the numeric sub-pipeline
num_pipe = full_pipeline.named_steps['preprocessor'].named_transformers_['num']
print("Fill values learned:", num_pipe.named_steps[________].statistics_)
```

<details>
<summary><strong>Select for answer</strong></summary>

```python
preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

full_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier',   RandomForestClassifier(n_estimators=100, random_state=42))
])

full_pipeline.fit(X_train, y_train)

print(f'Test accuracy: {full_pipeline.score(X_test, y_test):.3f}')

num_pipe = full_pipeline.named_steps['preprocessor'].named_transformers_['num']
print('Fill values learned:', num_pipe.named_steps['imputer'].statistics_)
```

Accessing nested components follows the chain: `full_pipeline.named_steps['preprocessor']` returns the `ColumnTransformer`; `.named_transformers_['num']` returns the numeric sub-pipeline; `.named_steps['imputer']` returns the fitted `SimpleImputer`; `.statistics_` holds the fill values learned from training data. This inspection pattern is useful for auditing what the pipeline actually learned.

</details>

### Exercise 7.3 Fix the bug

The pipeline is trained and then the test data is preprocessed manually before calling predict — breaking the pipeline's purpose. Fix it.

```python
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model',  LogisticRegression(max_iter=5000))
])
pipe.fit(X_train, y_train)

# Manual preprocessing — bug
scaler = StandardScaler()
X_test_scaled = scaler.fit_transform(X_test)
y_pred = pipe.predict(X_test_scaled)
print(f"Accuracy: {(y_pred == y_test).mean():.3f}")
```

<details>
<summary><strong>Select for answer</strong></summary>

```python
y_pred = pipe.predict(X_test)
print(f'Accuracy: {(y_pred == y_test).mean():.3f}')
```

The pipeline contains its own fitted scaler. Passing `X_test_scaled` (scaled by a separately fitted scaler) into `pipe.predict()` means the data is scaled twice — once by the manual scaler and once by the pipeline's internal scaler. Always pass raw, unprocessed data to a fitted pipeline and let it handle all preprocessing internally.

</details>

### Exercise 7.4 Predict the output

What will `pipe.predict()` return for the new sample, and why does it not matter that the new data has very different values from the training data?

```python
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model',  LogisticRegression(max_iter=300))
])
pipe.fit(X_train, y_train)

# New flower measurement (sepal length, sepal width, petal length, petal width)
new_sample = [[5.1, 3.5, 1.4, 0.2]]
prediction = pipe.predict(new_sample)
print(prediction)
print(pipe.predict_proba(new_sample).round(3))
```

<details>
<summary><strong>Select for answer</strong></summary>

```python
# [0]  (Iris setosa — the values match setosa characteristics)
# probabilities close to [1.0, 0.0, 0.0] for setosa, versicolor, virginica
```

The pipeline automatically applies the training scaler's statistics (means and stds learned from `X_train`) to the new sample before passing it to the classifier. There is no need to manually scale new data — the pipeline ensures the same transformation is applied consistently. `predict_proba` returns class probabilities; `predict` returns the class with the highest probability.

</details>

### Exercise 7.5 Adapt and extend

The example below saves and loads a simple pipeline using `joblib`:

```python
import joblib
joblib.dump(pipe, 'model.pkl')
loaded = joblib.load('model.pkl')
print(loaded.score(X_test, y_test))
```

Build a complete pipeline for the breast cancer dataset (all features are numeric), save it with a descriptive filename that includes the test accuracy, load it back, and confirm the loaded pipeline produces identical predictions to the original.

```python
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
import joblib
import numpy as np

X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Your code here
```

<details>
<summary><strong>Select for answer</strong></summary>

```python
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model',  LogisticRegression(max_iter=5000))
])
pipe.fit(X_train, y_train)

accuracy = pipe.score(X_test, y_test)
filename = f'breast_cancer_pipeline_acc{accuracy:.3f}.pkl'
joblib.dump(pipe, filename)
print(f'Saved: {filename}')

loaded = joblib.load(filename)
original_preds = pipe.predict(X_test)
loaded_preds   = loaded.predict(X_test)

print('Predictions identical:', np.array_equal(original_preds, loaded_preds))
print(f'Loaded accuracy: {loaded.score(X_test, y_test):.3f}')
```

`joblib.dump` serialises the entire pipeline — including the fitted scaler's learned statistics and the model's trained weights — into a single file. Loading it with `joblib.load` restores the pipeline to its exact fitted state. Including the accuracy in the filename is a simple but effective way to track which version of a model a file corresponds to.

</details>

---

## Well done for completing Unit 5 exercises!

If any section felt difficult, revisit the relevant part of the Unit 5 reading material before moving on.

**Key reminders:**

- Always **split first, then fit** — fitting any transformer on the full dataset before splitting leaks test information into the training process
- Call `.fit_transform()` only on training data; call `.transform()` on validation and test data using the statistics already learned from training
- `stratify=y` in `train_test_split` preserves the class ratio in both splits — essential when classes are imbalanced
- `StandardScaler` produces mean ≈ 0 and std = 1; `MinMaxScaler` squeezes values into [0, 1]; `RobustScaler` uses the median and IQR, making it resistant to outliers
- `SimpleImputer` fills missing values using statistics learned from training data — use `strategy='median'` for numeric columns with outliers, `strategy='most_frequent'` for categorical columns
- Use `OrdinalEncoder` only for features with a genuine order (e.g. Low < Medium < High); use `OneHotEncoder` for nominal categories with no natural ranking
- `OneHotEncoder(handle_unknown='ignore')` silently produces an all-zeros row for categories not seen during training — without this, unseen categories raise a `ValueError`
- A `Pipeline` applies preprocessing steps automatically during both `.fit()` and `.predict()` — never preprocess new data manually before passing it to a fitted pipeline
- `pipe.named_steps['step_name']` accesses a fitted step object; `preprocessor.named_transformers_['name']` accesses a fitted transformer inside a `ColumnTransformer`
- `joblib.dump` / `joblib.load` serialises the entire fitted pipeline — the saved file captures all learned parameters and can be used directly for inference without retraining